# Method 2 — Bi-Encoder (Kaggle T4)

Phase 2 của `docs/method2_plan.md`: train 2 vòng, pre-compute index, hiệu chỉnh ngưỡng. Ngân sách ~4h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [ ]:
# ===== Cell 0: dò dataset + HF cache =====
# PHẢI chạy trước mọi import transformers: thư viện chốt cache lúc import,
# set HF_HOME sau đó thì không còn tác dụng.
import os
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')


def _dirs_within(base: Path, max_depth: int = 4):
    """Mọi thư mục tới độ sâu `max_depth`, bỏ qua `hub/` cho nhanh."""
    frontier, seen = [base], []
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                children = [c for c in d.iterdir() if c.is_dir() and c.name != 'hub']
            except (PermissionError, OSError):
                continue
            seen.extend(children)
            nxt.extend(children)
        frontier = nxt
    return seen


def find_root(marker: str, label: str) -> Path:
    """Tìm thư mục chứa `marker`.

    Kaggle mount theo dạng /kaggle/input/datasets/<user>/<ds>/<ds>/, và số tầng
    đổi theo cách upload. Dò theo marker thì không phải hardcode username hay
    độ sâu — upload kiểu nào cũng tìm ra.
    """
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT):
        if (d / marker).exists():
            return d
    raise SystemExit(
        f'Không tìm thấy {label}: không thư mục nào dưới {INPUT_ROOT} có {marker}.\n'
        'Kiểm tra đã Add đủ 3 dataset ở sidebar Input chưa.'
    )


SRC_ROOT = find_root('src/models/preflight.py', 'dataset src')
DATA_ROOT = find_root('method2/manifest.json', 'dataset data')
HF_HOME = find_root('hub/models--BAAI--bge-m3', 'dataset hf-cache')

print('SRC :', SRC_ROOT)
print('DATA:', DATA_ROOT)
print('HF  :', HF_HOME)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

# Có thư mục model chưa đủ — thiếu file trọng số thì lỗi chỉ lộ ra lúc nạp
# model, sau khi đã tốn thời gian cài đặt và copy.
for name in ('models--BAAI--bge-m3', 'models--xlm-roberta-base'):
    weights = [
        f for f in (HF_HOME / 'hub' / name).rglob('*')
        if f.is_file() and f.suffix in ('.safetensors', '.bin') and f.stat().st_size > 10**8
    ]
    assert weights, f'{name}: không có file trọng số > 100 MB'
    print(f'  {name}: {max(f.stat().st_size for f in weights) / 1024**3:.2f} GB')
print('\nHF cache OK')


In [ ]:
# ===== Cell 1: cài package — PIN version =====
# Ba package chính quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
#
# Ưu tiên wheelhouse offline: notebook Kaggle có thể không bật internet,
# và transformers 5.x cần huggingface_hub 1.x + tokenizers 0.22.x — bản
# trong image gần như chắc chắn cũ hơn nên `import transformers` sẽ vỡ.
WHEELS = None
for candidate in _dirs_within(INPUT_ROOT) + [INPUT_ROOT]:
    if (candidate / 'wheels').is_dir() and any((candidate / 'wheels').glob('*.whl')):
        WHEELS = candidate / 'wheels'
        break

PACKAGES = ['transformers==5.15.1', 'sentence-transformers==6.0.0',
            'peft==0.20.0', 'datasets==5.0.1', 'jsonschema']
if WHEELS is not None:
    print('cài offline từ', WHEELS, f'({len(list(WHEELS.glob("*.whl")))} wheel)')
    !pip install -q --no-index --find-links {WHEELS} {' '.join(PACKAGES)}
else:
    print('không có wheelhouse — cài từ PyPI (cần internet)')
    !pip install -q {' '.join(PACKAGES)}


In [ ]:
# ===== Cell 1b: xác nhận import được và đúng version =====
# Cài xong không có nghĩa là dùng được: thiếu một dep thì lỗi chỉ lộ ra
# giữa lúc train. Kiểm ngay tại đây cho rẻ.
import importlib

EXPECTED = {
    'transformers': '5.15.1',
    'sentence_transformers': '6.0.0',
    'peft': '0.20.0',
}
for module_name, expected in EXPECTED.items():
    module = importlib.import_module(module_name)
    actual = getattr(module, '__version__', '?')
    print(f'  {module_name:24s} {actual}')
    assert actual == expected, f'{module_name}: {actual} ≠ {expected}'

import datasets, huggingface_hub, tokenizers
print(f'  {"datasets":24s} {datasets.__version__}')
print(f'  {"huggingface_hub":24s} {huggingface_hub.__version__}')
print(f'  {"tokenizers":24s} {tokenizers.__version__}')

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version.
import torch

free, total = torch.cuda.mem_get_info()
print(f'\n  torch {torch.__version__}')
print(' ', torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('  bf16 supported:', torch.cuda.is_bf16_supported())
# Kỳ vọng: Tesla T4, ~15.0 GB free, bf16 = False → mọi config dùng fp16.


In [ ]:
# ===== Cell 2: copy code + data vào /kaggle/working =====
# Dataset chỉ đọc, mà code ghi checkpoint và dùng đường dẫn tương đối, nên
# phải copy sang thư mục ghi được. Dùng path đã dò ở Cell 0.
import shutil

WORK = Path('/kaggle/working')
for name in ('src', 'configs'):
    target = WORK / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(SRC_ROOT / name, target)

# Dataset data bắt đầu thẳng bằng method2/ custom_vi/ benchmark_vi/ (KHÔNG có
# tầng `data/`), còn code tham chiếu `data/method2/...` → copy vào data/.
data_dir = WORK / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir)
data_dir.mkdir(parents=True)
for child in DATA_ROOT.iterdir():
    dest = data_dir / child.name
    shutil.copytree(child, dest) if child.is_dir() else shutil.copy2(child, dest)

%cd /kaggle/working

import json, glob, sys
sys.path.insert(0, '/kaggle/working')
# HF_HOME đã set ở Cell 0, kế thừa sang mọi tiến trình con `!python`.

print('src    :', sorted(p.name for p in (WORK / 'src').iterdir()))
print('data   :', sorted(p.name for p in data_dir.iterdir()))


In [ ]:
# ===== Cell 3: kiểm tra bản copy TRƯỚC khi preflight =====
# Preflight kiểm tra tính đúng đắn của dữ liệu; cell này kiểm tra bước copy —
# tách ra để khi hỏng thì biết ngay là hỏng ở đâu.
REQUIRED = [
    'data/method2/decontamination.json',
    'data/method2/manifest.json',
    'data/method2/tool_pool.json',
    'data/method2/biencoder/train.jsonl',
    'data/method2/biencoder/val.jsonl',
    'data/method2/biencoder/pairs_stats.json',
    'data/method2/crossencoder/train.jsonl',
    'data/method2/crossencoder/val.jsonl',
    'data/method2/label_stats.json',
    'data/custom_vi/v1/test_seen.jsonl',
    'data/benchmark_vi/test.jsonl',
    'configs/method2/biencoder.yaml',
    'configs/method2/pinned_versions.json',
    'src/models/preflight.py',
]
missing = []
for rel in REQUIRED:
    path = WORK / rel
    if path.exists() and path.stat().st_size > 0:
        print(f'  {path.stat().st_size / 1024**2:8.2f} MB  {rel}')
    else:
        missing.append(rel)
        print(f'  {"THIẾU":>11}  {rel}')
assert not missing, f'Copy chưa đủ: {missing}'

# import được thì mới chắc src/ copy nguyên vẹn.
import importlib

importlib.import_module('src.models.preflight')
manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('\nsnapshot commit:', manifest.get('git_commit'))
print('copy OK')


## Pre-flight — cổng fail-closed TRƯỚC mọi training

```
decontamination.json tồn tại
        ↓
SHA-256 == manifest.json
        ↓
overlap train/val/test == 0
        ↓
unseen positive leakage == 0
        ↓
package versions khớp bản đã pin
        ↓
CHO PHÉP TRAIN
```

Thiếu file hoặc hash lệch → job dừng ngay, **không rebuild tự động**. Nếu
experiment chính tự dựng lại index từ dữ liệu đang có trên máy thì ta mất
đúng thứ cần đảm bảo: bằng chứng model được train trên đúng split đã kiểm
định. Rebuild là lệnh preprocessing riêng, chạy ở local rồi upload lại:
`python -m src.models.sources decontaminate && python -m src.models.sources manifest`

Vì sao `val ∩ test` là rủi ro nặng nhất: dù không train trên query đó, việc
chọn checkpoint/hyperparameter bằng val vẫn khiến metric test lạc quan hơn
thực tế. `data/benchmark_vi` **giữ nguyên** — decontamination nằm ở tầng
dataset của Method 2 nên bốn method vẫn được đánh giá trên cùng một tập test.


In [ ]:
# Exit code != 0 → dừng notebook, không chạy tiếp cell training nào.
!python -m src.models.preflight \
    --config configs/method2/biencoder.yaml \
    --require-gpu T4 \
    --output results/method2/preflight.json

preflight = json.load(open('results/method2/preflight.json', encoding='utf-8'))
assert preflight['passed'], f"Preflight KHÔNG ĐẠT: {preflight['failures']}"
print('preflight PASS —', len(preflight['checks']), 'check')


In [ ]:
# Số liệu split để đối chiếu bằng mắt trước khi tiêu giờ GPU.
stats = json.load(open('data/method2/biencoder/pairs_stats.json', encoding='utf-8'))
decon = stats['decontamination']

print('unique query/split :', stats['unique_queries_per_split'])
print('positive pairs     :', stats['n_positive_pairs'])
print('negative samples   :', stats['n_negative_samples'])
print('query trùng split  :', decon['n_overlapping_queries'], decon['overlapping_queries'])
print('sample bị loại     :', decon['rows_dropped_total'], decon['rows_dropped_by_transition'])
print('overlap còn lại    :', stats['split_overlap_after'])


## Round 1 — train với hard negative round-0

`CachedMultipleNegativesRankingLoss` (GradCache) cho effective batch 256 với
mini_batch 8. Gradient accumulation **không** thay thế được: nó chỉ chia nhỏ
update chứ không làm tăng số in-batch negative.


In [ ]:
RUN = '/kaggle/working/artifacts/method2/biencoder/run01'
resume = sorted(glob.glob(f'{RUN}/checkpoint-*'))[-1] if glob.glob(f'{RUN}/checkpoint-*') else None
print('resume from:', resume)

!python -m src.models.biencoder.train train \
    --config configs/method2/biencoder.yaml \
    --output-dir {RUN} \
    --resume-from {resume}


## Round 2 — mine hard negatives rồi train lại **từ base**

Lấy tool sai nhưng xếp hạng cao (bỏ top-1 để tránh false negative). Round 2
train lại từ checkpoint gốc, không train tiếp từ round 1.


In [ ]:
!python -m src.models.biencoder.train mine \
    --config configs/method2/biencoder.yaml \
    --model {RUN}/final

RUN2 = '/kaggle/working/artifacts/method2/biencoder/run02'
!sed -i 's#biencoder/train.jsonl#biencoder/train_mined.jsonl#' configs/method2/biencoder.yaml
!python -m src.models.biencoder.train train \
    --config configs/method2/biencoder.yaml --output-dir {RUN2}


## Pre-compute index + hiệu chỉnh ngưỡng trên **val**


In [ ]:
!python -m src.models.biencoder.index \
    --config configs/method2/biencoder.yaml --model {RUN2}/final

# τ và τ_call CHỈ được hiệu chỉnh trên val, rồi freeze trước khi chạy test.
!python -m src.models.biencoder.evaluate calibrate \
    --config configs/method2/biencoder.yaml \
    --model {RUN2}/final \
    --pairs data/method2/biencoder/val.jsonl \
    --output {RUN2}/thresholds.json


## Gate để qua Phase 3

| Metric | Tập | Ngưỡng |
|---|---|---|
| Recall@1 | custom `val_seen` | ≥ 0.90 |
| Recall@1 | custom `val_unseen` | ≥ 0.75 |
| Recall@5 | custom `val_unseen` | ≥ 0.92 |
| Negative Recall @ τ | custom val negative | ≥ 0.80 |

Không đạt → thử theo thứ tự: (a) thêm param name vào document text,
(b) tăng hard negative lên 8, (c) đổi sang `AITeamVN/Vietnamese_Embedding`,
(d) full fine-tune thay LoRA.


In [ ]:
!python -m src.models.biencoder.evaluate evaluate \
    --config configs/method2/biencoder.yaml \
    --model {RUN2}/final \
    --pairs data/method2/biencoder/val.jsonl \
    --output results/method2/metrics/biencoder_val.json

report = json.load(open('results/method2/metrics/biencoder_val.json', encoding='utf-8'))
for slice_name, metrics in report['by_source_key'].items():
    print(slice_name, {k: v for k, v in metrics.items() if 'recall@' in k or k == 'mrr'})


## Run manifest — chốt lại toàn bộ mục audit

Train xong mà không audit được thì coi như chưa train. Cell này gom: commit
SHA (kèm cờ dirty), config YAML thực tế, fingerprint dataset + tool pool,
query counts và overlap theo split, số positive/negative pair, checkpoint,
best step + metric đã dùng để chọn, VRAM peak, thời lượng train, và
Recall@1/@5/@10 + MRR. Thiếu mục nào thì `audit_complete.missing` liệt kê ra.

`checkpoint_selection.available_metrics` cho biết tên metric thật của
`InformationRetrievalEvaluator` ở phiên bản đang chạy — khai vào
`train.metric_for_best_model` cho lần chạy sau.


In [ ]:
!python -m src.models.run_manifest \
    --run-dir {RUN2} \
    --config configs/method2/biencoder.yaml \
    --stage biencoder \
    --report retrieval=results/method2/metrics/biencoder_val.json

manifest = json.load(open(f'{RUN2}/run_manifest.json', encoding='utf-8'))
missing = manifest['audit_complete']['missing']
print('thiếu:', missing or 'không thiếu mục nào')
print('Recall/MRR:', manifest.get('retrieval_gate', {}).get('metrics'))
print('VRAM peak MB:', manifest['train']['peak_vram_mb'])
print('thời lượng (giờ):', manifest['train']['duration_hours'])
print('chọn checkpoint:', manifest['train']['checkpoint_selection'])
assert not missing, f'Chưa đủ artifact để audit: {missing}'


In [ ]:
# ===== Lưu artifact =====
# Kaggle chỉ giữ /kaggle/working (20GB). Nén để tải về hoặc làm Dataset mới.
!tar czf /kaggle/working/biencoder_run.tar.gz -C /kaggle/working/artifacts/method2 .
!du -h /kaggle/working/biencoder_run.tar.gz
